In [4]:
import gymnasium as gym
import numpy as np
import torch

from collections import deque
import torch.optim as optim
from torch.distributions import Categorical

In [ ]:
def reinforce(env, policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # Help us to calculate the score during the training
    scores_deque = deque(maxlen=100)
    scores = []
    # Line 3 of pseudocode
    for i_episode in range(1, n_training_episodes+1):
        saved_log_probs = []
        rewards = []
        state, info = env.reset()
        # Line 4 of pseudocode
        for t in range(max_t):
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            state, reward, terminated, truncated, info = env.step(action)
            rewards.append(reward)
            done = terminated or truncated
            if done:
                break
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Line 6 of pseudocode: calculate the return
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        # Compute the discounted returns at each timestep,
        # as the sum of the gamma-discounted return at time t (G_t) + the reward at time t

        # In O(N) time, where N is the number of time steps
        # (this definition of the discounted return G_t follows the definition of this quantity
        # shown at page 44 of Sutton&Barto 2017 2nd draft)
        # G_t = r_(t+1) + r_(t+2) + ...

        # Given this formulation, the returns at each timestep t can be computed
        # by re-using the computed future returns G_(t+1) to compute the current return G_t
        # G_t = r_(t+1) + gamma*G_(t+1)
        # G_(t-1) = r_t + gamma* G_t
        # (this follows a dynamic programming approach, with which we memorize solutions in order
        # to avoid computing them multiple times)

        # This is correct since the above is equivalent to (see also page 46 of Sutton&Barto 2017 2nd draft)
        # G_(t-1) = r_t + gamma*r_(t+1) + gamma*gamma*r_(t+2) + ...


        ## Given the above, we calculate the returns at timestep t as:
        #               gamma[t] * return[t] + reward[t]
        #
        ## We compute this starting from the last timestep to the first, in order
        ## to employ the formula presented above and avoid redundant computations that would be needed
        ## if we were to do it from first to last.

        ## Hence, the queue "returns" will hold the returns in chronological order, from t=0 to t=n_steps
        ## thanks to the appendleft() function which allows to append to the position 0 in constant time O(1)
        ## a normal python list would instead require O(N) to do this.
        for t in range(n_steps)[::-1]:
            disc_return_t = (returns[0] if len(returns)>0 else 0)
            returns.appendleft(  gamma * disc_return_t + rewards[t] ) # TODO: complete here

        ## standardization of the returns is employed to make training more stable
        eps = np.finfo(np.float32).eps.item()

        ## eps is the smallest representable float, which is
        # added to the standard deviation of the returns to avoid numerical instabilities
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # Line 7:
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        # Line 8: PyTorch prefers gradient descent
        optimizer.zero_grad()

        policy_loss.backward()
        
        optimizer.step()

        if i_episode % print_every == 0:
            print('Episode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_deque)))

    return scores
    

In [ ]:
from model import Policy

env_id = "CartPole-v1"
env = gym.make(env_id)

observation, info = env.reset()

learning_rate = 5e-5
policy = Policy()
adamw = optim.AdamW(policy.parameters(), lr=learning_rate)


print(observation, info)
print("Observation Shape")

# cart position, velocity, pole angle, angular velocity
print(env.observation_space.sample)

scores = reinforce(
    env=env,
    policy=policy,
    optimizer=adamw,
    n_training_episodes=10000,
    max_t=25,
    gamma=0.99, #discount factor
    print_every=100
)

[ 0.02914309  0.00760707 -0.03414939 -0.02432322] {}
Observation Shape
<bound method Box.sample of Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)>
Episode 100	Average Score: 19.50
Episode 200	Average Score: 18.66
Episode 300	Average Score: 18.78
Episode 400	Average Score: 19.39
Episode 500	Average Score: 17.71
Episode 600	Average Score: 18.65
Episode 700	Average Score: 19.12
Episode 800	Average Score: 18.99
Episode 900	Average Score: 19.09
Episode 1000	Average Score: 18.20
Episode 1100	Average Score: 18.27
Episode 1200	Average Score: 17.51
Episode 1300	Average Score: 19.26
Episode 1400	Average Score: 18.10
Episode 1500	Average Score: 18.15
Episode 1600	Average Score: 18.78
Episode 1700	Average Score: 19.55
Episode 1800	Average Score: 18.27
Episode 1900	Average Score: 19.07
Episode 2000	Average Score: 18.67
Episode 2100	Average Score: 18.51
Episode 2200	Average Score: 18.30
Episode 2300	Average Score: 18.20
Episode 24

In [7]:
def evaluate_agent(env, max_steps, n_eval_episodes, policy):
    """
    Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
    :param env: The evaluation environment
    :param n_eval_episodes: Number of episode to evaluate the agent
    :param policy: The Reinforce agent
    """
    episode_rewards = []
    for episode in range(n_eval_episodes):
        state, info = env.reset()
        done = False
        total_rewards_ep = 0

        for _ in range(max_steps):
            action, _ = policy.act(state)
            new_state, reward, terminated, truncated, info = env.step(action)
            total_rewards_ep += reward
            done = terminated or truncated

            if done:
                break
            state = new_state
        episode_rewards.append(total_rewards_ep)
    mean_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)

    return mean_reward, std_reward

In [8]:
rewards, std_reward = evaluate_agent(
    env,
    max_steps = 100,
    n_eval_episodes=10,
    policy=policy 
)

print(rewards, std_reward)

20.7 10.752209075348192


In [9]:
import imageio

def record_video(env, policy, out_directory, fps=30):
    """
    Generate a replay video of the agent.

    Gymnasium expects the environment to be created with ``render_mode="rgb_array"``
    instead of passing ``mode`` to ``render()``.
    """
    render_env = gym.make(env.spec.id, render_mode="rgb_array")

    images = []
    done = False
    state, info = render_env.reset()
    images.append(render_env.render())

    while not done:
        action, _ = policy.act(state)
        state, reward, terminated, truncated, info = render_env.step(action)
        done = terminated or truncated
        images.append(render_env.render())

    render_env.close()
    imageio.mimsave(out_directory, [np.array(img) for img in images], fps=fps)

In [11]:
record_video(
    env,
    policy,
    out_directory="eval.gif",
    fps=60
)